In [0]:
from pyspark.sql import functions as F

In [0]:
raw_aircraft_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", "'") \
    .option("escape", "'") \
    .option("multiLine", "true") \
    .load("/Volumes/airspace_pulse/bronze/static_data/aircraft/aircraft-database.csv")


In [0]:
display(raw_aircraft_df.limit(20))

In [0]:
cleaned_dim_aircraft = raw_aircraft_df \
    .filter(
        (F.col("icao24").isNotNull()) &
        (F.col("icao24") != "000000")
    ) \
    .select(
        F.lower(F.trim(F.col("icao24"))).alias("icao24"),
        # registration
        F.when(F.trim(F.col("registration")).isin("", "-", "null"), None)
        .otherwise(F.upper(F.trim(F.col("registration"))))
        .alias("registration"),
        # manufacturer name
        F.coalesce(
            F.when(F.trim(F.col("manufacturerName")).isin("", "-", "null"), None).otherwise(F.trim(F.col("manufacturerName"))),
            F.when(F.trim(F.col("manufacturerIcao")).isin("", "-", "null"), None).otherwise(F.trim(F.col("manufacturerIcao")))
        ).alias("manufacturer"),
        # model serise
        F.when(F.trim(F.col("model")).isin("unknow", "unknown", "-", ""), None)
        .otherwise(F.trim(F.col("model")))
        .alias("model_serise"),
        # type code 
        F.when(F.trim(F.col("typecode")).isin("", "-", "null"), None)
        .otherwise(F.upper(F.trim(F.col("typecode"))))
        .alias("typecode"),
        # category description
        F.when(F.trim(F.col("categoryDescription")).isin("", "-", "null"), None)
        .otherwise(F.trim(F.col("categoryDescription")))
        .alias("category_description"),
        # Operator / Owner Name
        F.coalesce(
            F.when(F.trim(F.col("operator")).isin("", "-", "null"), None).otherwise(F.trim(F.col("operator"))),
            F.when(F.trim(F.col("owner")).isin("", "-", "null"), None).otherwise(F.trim(F.col("owner")))
        ).alias("operator_name"),
        # Operator Codes
        F.when(F.trim(F.col("operatorIcao")).isin("", "-", "null"), None)
         .otherwise(F.upper(F.trim(F.col("operatorIcao"))))
         .alias("operator_icao"),
        # Registration Country
        F.when(F.trim(F.col("country")).isin("", "-", "null"), None)
         .otherwise(F.trim(F.col("country")))
         .alias("registered_country")
    ) \
    .dropDuplicates(["icao24"])


In [0]:
# 3. Write to Silver Delta Lake
cleaned_dim_aircraft.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airspace_pulse.silver.dim_aircraft")

In [0]:
%sql
SELECT * FROM airspace_pulse.silver.dim_aircraft WHERE icao24 = 'a3401a'